# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2 rangeland management dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

The Croissant metadata provides a description of available record sets and fields. We'll list all available record sets and, for each, enumerate its fields and columns, referencing all entities by their `@id`.

In [ ]:
# List all available record sets and their fields by @id
if not hasattr(metadata, 'record_sets'):
    print("No record sets found in metadata.")
else:
    for rs in metadata.record_sets:
        print(f"Record set: @id={rs.id} | name={getattr(rs, 'name', '')}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for field in rs.fields:
                print(f"    @id={field.id} | name={getattr(field, 'name', '')}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    @id={col.id} | name={getattr(col, 'name', '')}")
        print("---")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s for precise referencing.

In [ ]:
# List all record set IDs for extraction
record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns from the first available record set (if any)
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Columns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on criteria, normalizing numeric fields, and grouping/categorizing data. All field references are made using their Croissant `@id`.

In [ ]:
# Perform EDA on the first record set if available
import numpy as np

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Performing EDA on record set: {record_set_id}")
    numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Example: filter above mean
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical column if exists
        categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        for col in categorical_columns:
            if col != numeric_field_id:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No numeric fields found in this record set for EDA.")
else:
    print("No data extracted for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore a dataset containing ordered logistic regression results for knowledge adoption in rangeland management in Northern Kenya.

- We loaded dataset metadata, explored available record sets and fields using their `@id` for precise referencing.
- We loaded records for each record set, performed filtering, normalization, and grouping, and visualized key numeric variables.

This provides a starting point for deeper analysis of the data and helps ensure reproducibility by referencing all dataset entities via their Croissant `@id`s.